# Projeto IIA

Grupo AQ

- Juniper Wilson 61795
- Jaime Sousa 58171
- Miguel Zhang 61829

Contribuições do grupo:

- Juniper 33%
- Jaime 33%
- Miguel 33%

## Estrategia

Reparem no seguinte acerca dos problemas:

Um de classificação: 
O1

Outros dois de regressão:
O2 e O3

O objetivo O4 é resolvido no notebook da TP08: Feature selection

O nosso metodo de trabalho deve passar por:

1) Induzir os dados ausentes (Data Inputing)
2) Reduzir as features triviais usando Spearman (Feature Selection)
3) Construir classificadores/regressores demonstrados nas tps
4) Pontuar e escolher os classificadores
5) Otimizando os seus hyperparametros

In [3]:

import sys
from pathlib import Path

# Make project root importable regardless of notebook working directory
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sklearn.feature_selection import SequentialFeatureSelector 
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler

# Scikit-learn: models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVC
from sklearn.svm import LinearSVR
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor

# Scikit-learn: model selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split

# Scikit-learn: metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import mean_squared_error
from sklearn.metrics import precision_score
from sklearn.metrics import r2_score
from sklearn.metrics import recall_score
from sklearn.metrics import root_mean_squared_error

# Our methods
from src.load import load_spaceship_data

#path
data_path = "../data/spaceship_data.csv"

## Data Inputing

Vamos induzir os dados ausentes usando um inputer baseado em KNN

In [ ]:
# TODO: Execute imputation using KNNImputers
ki = KNNImputer(n_neighbors=5)

## Feature Selection

Usamos a correlação Spearman para descobrir:

O4:  Quais  são  os  atributos  mais  importantes  nos  modelos O1, O2, O3?

In [4]:
# O4.1: Spearman correlation for O1, O2 and O3 using load.py
import importlib
from src.spearman import spearman_with_target

# O1: classification target
spearman_o1 = spearman_with_target(data_path, "Transported")
print("O1: Spearman Correlation with Target Transported:")
print(spearman_o1)
print("\n")

# O2: regression target
spearman_o2 = spearman_with_target(data_path, "FoodCourt")
print("O2: Spearman Correlation with Target FoodCourt:")
print(spearman_o2)
print("\n")

# O3: regression target, filtered to passengers that were transported
spearman_o3 = spearman_with_target(data_path, "Age", filter_col="Transported", filter_value=True)
print("O3: Spearman Correlation with Target Age (Transported=True only):")
print(spearman_o3)
print("\n")

print("Low correlation means the feature is not strongly related to the target. Consider dropping it.")

O1: Spearman Correlation with Target Transported:
              correlation        p_value
CryoSleep        0.458955   0.000000e+00
RoomService     -0.367189  6.087397e-221
Spa             -0.363072  1.060938e-215
VRDeck          -0.344708  3.033668e-193
ShoppingMall    -0.222237   1.480313e-78
FoodCourt       -0.176329   1.124358e-49
HomePlanet       0.138180   5.426828e-31
Destination     -0.101958   1.554600e-17
Age             -0.060485   4.473372e-07
Cabin           -0.056653   2.275076e-06
VIP             -0.033039   5.862320e-03
Name            -0.016093   1.796545e-01
PassengerId      0.013858   2.479049e-01


O2: Spearman Correlation with Target FoodCourt:
              correlation        p_value
CryoSleep       -0.524488   0.000000e+00
VRDeck           0.495686   0.000000e+00
Spa              0.465011   0.000000e+00
Cabin           -0.272895  5.183831e-119
Age              0.201339   1.602951e-64
ShoppingMall     0.181943   7.950937e-53
Transported     -0.176329   1.124358e-4

c:\Users\jsousa\OneDrive - FCT\Documents\1.Faculdade\2.Code\Projeto_IAA26_ST\src\spearman.py:63: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = stats.spearmanr(series, y_s)


### Conclusão

Dadas as baixas correlações, decidimos dropar as colunas:

O1: PassengerId, Name

O2: PassengerId, Destination, HomePlanet, Name

O3: CryoSleep, PassengerId, Name, Destination

Obs: Dropa-se também Transported em O3 dada a natureza do problema


In [5]:
# Load complete dataframe
import src.load as load_module
import pandas as pd

raw_df = load_module.load_spaceship_data_as_df(data_path)

# O1: Drop PassengerId and Name
df_o1 = raw_df.drop(columns=["PassengerId", "Name"])

# O2: Drop PassengerId, Name, Destination, HomePlanet
df_o2 = raw_df.drop(columns=["PassengerId", "Name", "Destination", "HomePlanet"])

# O3: Filter to Transported=True, then drop PassengerId, Name, Destination, CryoSleep, Transported
df_o3 = raw_df[raw_df["Transported"] == True].drop(columns=["PassengerId", "Name", "Destination", "CryoSleep", "Transported"])

# Debugging: print shapes of the resulting dataframes
print(f"O1 shape: {df_o1.shape}")
print(f"O2 shape: {df_o2.shape}")
print(f"O3 shape: {df_o3.shape}")

O1 shape: (6954, 12)
O2 shape: (6954, 10)
O3 shape: (3502, 9)


# Exemplo de um classificador

In [ ]:
# Load the raw data
import importlib
import sys
from pathlib import Path

# Make project root importable regardless of notebook working directory
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.statistics import present_classification_statistics, present_simple_statistics
import src.load as load_module
importlib.reload(load_module)

raw_df = load_module.load_spaceship_data_as_df(str(project_root / "data" / "spaceship_data.csv"))
y = raw_df["Transported"].astype(int)
X = raw_df.drop(columns=["Transported", "PassengerId", "Name"], errors='ignore')

# Basic preprocessing so the tree can train on numeric data
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode().iloc[0])

X = pd.get_dummies(X, drop_first=True)

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Train DecisionTreeClassifier
clf = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=42)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Print metrics
present_classification_statistics(y_test, y_pred)

present_simple_statistics(y_test, y_pred)


=== DECISION TREE CLASSIFIER METRICS ===

Confusion Matrix:
[[486 230]
 [113 562]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.68      0.74       716
           1       0.71      0.83      0.77       675

    accuracy                           0.75      1391
   macro avg       0.76      0.76      0.75      1391
weighted avg       0.76      0.75      0.75      1391

Precision: 0.7619737501198814
Recall: 0.7534148094895758
F1-Score: 0.7522782018326476
Matthews Correlation Coefficient: 0.5161336297331891
R2: 0.0128
RMSE: 0.4966
Correlation Score: 0.5161 (p-value=1.516669e-95)
Maximum Error: 1.0000
Mean Absolute Error: 0.2466



# Problema de Classificação:

O1:  Prever  a  variável  Transported  dados  os  atributos  do  dataset 
fornecido. 

In [ ]:
#loads the data
import importlib
import load as load_module
importlib.reload(load_module)
spaceship_data = load_module.load_spaceship_data("data/spaceship_data.csv")
X, y, feature_names = load_module.load_spaceship_data("data/spaceship_data.csv")
#splits the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)





# Problemas de Regressão:

O2: Prever  o  valor  dispendido  pelos  passageiros  em  restaurantes 
(variável FoodCourt) 

In [ ]:
# O2 analysis (see O4.1 for Spearman correlations)

O3:  É  possível  prever  com  confiança  a  idade  dos  passageiros  que 
foram transportados? 

In [68]:
#O3